# 00b: AI-Assisted Programming for Hydrologists

In this notebook we introduce AI-assisted programming concepts and explore how large language model (LLM) based coding assistants can help -- and mislead -- hydrologists writing Python code.

**Learning objectives:**
- Understand at a high level how LLM-based coding assistants generate code
- Recognize the strengths *and* limitations of AI code generation for scientific work
- Identify common problems when AI writes hydrology code
- Learn four prompt patterns that improve AI output for hydrology tasks
- Practice evaluating AI-generated code against known ground truth

**Time estimate:** ~25 minutes

> **Note:** All AI-generated code examples in this notebook were captured ahead of time. No live AI API calls are made. This means the notebook works without any AI tool installed.

### Summary of Prompt Patterns

| Pattern | Key Elements | When to Use |
|---------|-------------|-------------|
| Data Transformations | Input/output types, column names, units | Processing NWIS data, unit conversions |
| Equation Implementations | Full equation, variable definitions, units, library | Theis, Darcy, water balance calculations |
| Plot Customizations | Library, labels with units, style, scale | Hydrographs, cross-sections, maps |
| File I/O Operations | Format, structure, encoding, library | Reading rdb files, shapefiles, NetCDF |

## 1. What Are LLM-Based Coding Assistants?

Tools like GitHub Copilot, Amazon Q Developer, and Claude are built on **large language models** (LLMs). At a high level:

1. **Training**: The model is trained on vast amounts of text and code from the internet, books, and documentation. It learns statistical patterns -- which tokens (words, symbols) are likely to follow other tokens.

2. **Prediction**: When you type a prompt or partial code, the model predicts the most likely continuation based on patterns it learned during training.

3. **Generation**: The model produces code one token at a time, each token conditioned on everything that came before it.

**The key insight:** These models are very good at *pattern matching* and *statistical prediction*. They do **not** understand physics, they do **not** run the code they write, and they have **no way to verify** that their output is scientifically correct.

Think of it like a coder who has memorized millions of code examples but has never actually worked in a hydrology lab or run a groundwater model.

## 2. Strengths and Limitations

### What AI coding assistants are good at

| Strength | Example |
|----------|--------|
| **Pattern completion** | Finishing a function once you start typing the signature |
| **Boilerplate generation** | Writing repetitive setup code (file I/O, plot formatting) |
| **Syntax help** | Remembering the exact argument names for `matplotlib` or `pandas` |
| **Code translation** | Converting a FORTRAN snippet to Python |
| **Documentation** | Generating docstrings and comments |

### What AI coding assistants are *not* good at

| Limitation | Why it matters for hydrology |
|-----------|-----------------------------|
| **No domain understanding** | The model does not know that hydraulic conductivity must be positive |
| **Hallucination** | It may invent API functions that do not exist, or cite papers that were never written |
| **Cannot verify physics** | It cannot check whether a computed drawdown value is physically reasonable |
| **Outdated training data** | It may use deprecated `flopy.modflow` syntax instead of current `flopy.mf6` |
| **Confident when wrong** | It presents incorrect code with the same confidence as correct code |

The bottom line: AI assistants are **tools**, not experts. You bring the domain knowledge; the AI helps with the syntax.

## 3. Worked Examples: When AI Gets It Wrong

The following three examples show real categories of errors that AI coding assistants produce when writing hydrology code. Each example is syntactically valid Python that runs without errors -- but produces **scientifically wrong** results.

### Example 1: Unit Conversion Error (cfs to cms)

A hydrologist asks an AI assistant:

> *"Write a Python function to convert streamflow from cubic feet per second (cfs) to cubic meters per second (cms)."*

The AI produces:

In [ ]:
# AI-GENERATED CODE (contains an error!)
def cfs_to_cms(flow_cfs):
    """Convert streamflow from cubic feet per second to cubic meters per second."""
    conversion_factor = 0.3048  # feet to meters
    return flow_cfs * conversion_factor

**What went wrong?**

The AI used the *linear* feet-to-meters conversion factor (0.3048) instead of the *cubic* conversion factor. Since we are converting a volume rate:

$$1 \text{ ft}^3 = (0.3048)^3 \text{ m}^3 = 0.0283168 \text{ m}^3$$

The correct factor is **0.0283168**, not 0.3048. The AI's code overestimates flow by roughly **10x**.

Let's verify:

In [ ]:
# Correct conversion
correct_factor = 0.0283168

# AI's wrong conversion
wrong_factor = 0.3048

test_flow_cfs = 100.0  # 100 cfs

print(f"Test flow: {test_flow_cfs} cfs")
print(f"AI result:      {test_flow_cfs * wrong_factor:.4f} cms  <-- WRONG")
print(f"Correct result: {test_flow_cfs * correct_factor:.4f} cms")
print(f"AI overestimates by a factor of {wrong_factor / correct_factor:.1f}x")

**Lesson:** AI models pattern-match on "feet to meters" and grab the most common conversion factor. They do not reason about dimensional analysis (length vs. volume).

### Example 2: Incorrect Theis Equation Implementation

A hydrologist asks:

> *"Write a Python function to compute drawdown using the Theis equation. Use scipy."*

The AI produces:

In [ ]:
import numpy as np
from scipy.special import exp1

# AI-GENERATED CODE (contains an error!)
def theis_drawdown_ai(Q, T, S, r, t):
    """Compute drawdown using the Theis equation.
    
    Parameters
    ----------
    Q : float
        Pumping rate [L^3/T]
    T : float
        Transmissivity [L^2/T]
    S : float
        Storativity [-]
    r : float
        Distance from pumping well [L]
    t : float
        Time since pumping began [T]
    
    Returns
    -------
    s : float
        Drawdown [L]
    """
    u = (r**2 * S) / (4.0 * T * t)
    # BUG: using -exp1(u) instead of exp1(u)
    # The AI confused the well function W(u) = E1(u) with -Ei(-u)
    s = (Q / (4.0 * np.pi * T)) * (-exp1(u))
    return s

**What went wrong?**

The Theis well function $W(u)$ is the exponential integral $E_1(u)$, available in `scipy.special` as `exp1(u)`. The correct formula is:

$$s = \frac{Q}{4\pi T} \cdot E_1(u)$$

The AI added a **negative sign**, computing $-E_1(u)$ instead of $E_1(u)$. This is a common confusion because some references express the well function as $W(u) = -E_i(-u)$ where $E_i$ is a different exponential integral. The `scipy.special.exp1` function already returns the correct $E_1(u)$ value directly.

Let's compare against the known answer from the Theis exercise (07a):

In [ ]:
from scipy.special import exp1 as W

# Correct implementation
def theis_drawdown_correct(Q, T, S, r, t):
    """Compute drawdown using the Theis equation (correct)."""
    u = (r**2 * S) / (4.0 * T * t)
    s = (Q / (4.0 * np.pi * T)) * W(u)
    return s

# Test parameters from the Theis exercise
Q = 4088.0    # pumping rate [m^3/day]
T = 1000.0    # transmissivity [m^2/day]
S = 3e-4      # storativity [-]
r = 1000.0    # distance [m]
t = 10.0      # time [days]

# Ground truth from 07a_Theis-exercise
ground_truth = 1.40636669

ai_result = theis_drawdown_ai(Q, T, S, r, t)
correct_result = theis_drawdown_correct(Q, T, S, r, t)

print(f"Ground truth:    s = {ground_truth:.8f} m")
print(f"Correct code:    s = {correct_result:.8f} m")
print(f"AI code (wrong): s = {ai_result:.8f} m")
print(f"\nThe AI result is NEGATIVE -- physically impossible for drawdown from pumping!")

**Lesson:** The AI produced a function with a correct docstring, correct variable names, and correct structure -- but the sign error means it computes *negative* drawdown (water levels rising from pumping), which is physically impossible. The code runs without any errors or warnings.

### Example 3: Physically Impossible Parameter Values

A hydrologist asks:

> *"Write a function that computes Darcy flux given hydraulic conductivity, and hydraulic gradient. Include a docstring."*

The AI produces:

In [ ]:
# AI-GENERATED CODE (accepts physically impossible inputs!)
def darcy_flux_ai(K, dh_dl):
    """Compute Darcy flux (specific discharge).
    
    Parameters
    ----------
    K : float
        Hydraulic conductivity [L/T]
    dh_dl : float
        Hydraulic gradient [L/L]
    
    Returns
    -------
    q : float
        Darcy flux [L/T]
    """
    return -K * dh_dl

The code looks clean and correct. But watch what happens when we pass in physically impossible values:

In [ ]:
# Negative hydraulic conductivity -- physically impossible!
q = darcy_flux_ai(K=-5.0, dh_dl=0.01)
print(f"Darcy flux with K = -5.0: q = {q} m/day")
print("No error raised! The AI happily accepted a negative K value.")
print()

# A safer version would validate inputs
def darcy_flux_safe(K, dh_dl):
    """Compute Darcy flux with input validation.
    
    Parameters
    ----------
    K : float
        Hydraulic conductivity [L/T]. Must be positive.
    dh_dl : float
        Hydraulic gradient [L/L]
    
    Returns
    -------
    q : float
        Darcy flux [L/T]
    
    Raises
    ------
    ValueError
        If K is not positive.
    """
    if K <= 0:
        raise ValueError(f"Hydraulic conductivity must be positive, got K={K}")
    return -K * dh_dl

# Now try the impossible value
try:
    darcy_flux_safe(K=-5.0, dh_dl=0.01)
except ValueError as e:
    print(f"Caught error: {e}")
    print("The safe version rejects unphysical inputs.")

**Lesson:** AI-generated code almost never includes physical constraint validation. The model does not know that hydraulic conductivity is a material property that must be positive, or that porosity must be between 0 and 1, or that transmissivity cannot be negative. **You** need to add these checks based on your domain knowledge.

## 4. Prompt Patterns for Hydrology

The quality of AI-generated code depends heavily on how you write your prompt. Here are four patterns that work well for hydrology programming tasks.

### Pattern 1: Describing Data Transformations

Tell the AI exactly what goes in and what comes out.

**Vague prompt:**
> *"Convert my streamflow data."*

**Better prompt:**
> *"Write a function that takes a pandas DataFrame with columns 'datetime' (string, format 'YYYY-MM-DD') and 'flow_cfs' (float, daily mean streamflow in cubic feet per second). Return a new DataFrame with a DatetimeIndex and a column 'flow_cms' containing the flow converted to cubic meters per second using the factor 0.0283168."*

The better prompt specifies:
- Input data structure (DataFrame with named columns)
- Input data types and formats
- Output data structure
- The exact conversion factor to use

### Pattern 2: Requesting Equation Implementations

Include the equation, define every variable, and specify units.

**Vague prompt:**
> *"Implement the Theis equation."*

**Better prompt:**
> *"Implement the Theis equation for drawdown: s = Q/(4*pi*T) * W(u), where u = r^2*S/(4*T*t). Use scipy.special.exp1 for the well function W(u). Parameters: Q = pumping rate in m^3/day, T = transmissivity in m^2/day, S = storativity (dimensionless), r = distance in meters, t = time in days. Return drawdown s in meters. Include a numpy-style docstring."*

The better prompt:
- Provides the full equation
- Names the specific scipy function to use
- Specifies units for every parameter
- Requests documentation

### Pattern 3: Asking for Plot Customizations

Be specific about labels, units, and style.

**Vague prompt:**
> *"Plot my streamflow data."*

**Better prompt:**
> *"Using matplotlib, create a hydrograph plot from a pandas Series with a DatetimeIndex (daily values) and streamflow in cms. Use a blue line, set the y-axis label to 'Streamflow (m\u00b3/s)', the x-axis label to 'Date', and add a title 'Daily Mean Streamflow'. Use a log scale for the y-axis. Add a horizontal line at the mean flow value in red with a dashed style."*

The better prompt specifies:
- The library to use
- Input data format
- Axis labels with units
- Specific style choices
- Additional plot elements

### Pattern 4: Requesting File I/O Operations

Specify the file format, encoding, and expected structure.

**Vague prompt:**
> *"Read my data file."*

**Better prompt:**
> *"Write a function that reads a USGS NWIS tab-separated rdb file. The file has comment lines starting with '#', a header line with column names, a format line starting with a data type code, then tab-separated data. Use pandas read_csv with sep='\\t' and comment='#'. Skip the format line (row index 0 after the header). Parse the 'datetime' column as dates. Return a DataFrame indexed by datetime."*

The better prompt:
- Names the specific file format
- Describes the file structure (comments, headers, format line)
- Specifies the library and key arguments
- Describes how to handle the format line

## 5. Interactive Exercise: Well Drawdown Calculation

In this exercise, you will write a prompt for an AI assistant to compute well drawdown, then evaluate the output against a known ground truth.

### The Task

Compute the drawdown at a distance of **500 m** from a pumping well after **5 days** of pumping, given:
- Pumping rate $Q$ = 2,000 m$^3$/day
- Transmissivity $T$ = 500 m$^2$/day
- Storativity $S$ = 1 $\times$ 10$^{-4}$

Use the Theis equation:

$$s = \frac{Q}{4\pi T} \cdot W(u), \quad u = \frac{r^2 S}{4 T t}$$

### Step 1: Write your prompt

In the cell below, write the prompt you would give to an AI coding assistant. Use the prompt patterns from Section 4. Think about what information the AI needs to get this right.

In [ ]:
# Write your prompt as a comment or string here:
my_prompt = """
YOUR PROMPT HERE
"""
print(my_prompt)

### Step 2: Evaluate this AI-generated response

Below is code that an AI assistant might produce in response to a well-written prompt. Run it and compare against the ground truth.

In [ ]:
import numpy as np
from scipy.special import exp1

# Pre-captured AI response to a well-structured prompt
def compute_drawdown(Q, T, S, r, t):
    """Compute drawdown using the Theis equation.
    
    Parameters
    ----------
    Q : float
        Pumping rate [m^3/day]
    T : float
        Transmissivity [m^2/day]
    S : float
        Storativity [-]
    r : float
        Distance from pumping well [m]
    t : float
        Time since pumping began [days]
    
    Returns
    -------
    s : float
        Drawdown [m]
    """
    u = (r**2 * S) / (4.0 * T * t)
    s = (Q / (4.0 * np.pi * T)) * exp1(u)
    return s

# Compute drawdown with the given parameters
Q = 2000.0   # m^3/day
T = 500.0    # m^2/day
S = 1e-4     # dimensionless
r = 500.0    # m
t = 5.0      # days

s = compute_drawdown(Q, T, S, r, t)
print(f"Computed drawdown: s = {s:.8f} m")

### Step 3: Verify against ground truth

Run the cell below to check the AI-generated result against the known correct answer.

In [ ]:
# Ground truth calculation
u_check = (r**2 * S) / (4.0 * T * t)
s_check = (Q / (4.0 * np.pi * T)) * exp1(u_check)

ground_truth_s = 1.72420422  # pre-computed correct value

print(f"Your result:   s = {s:.8f} m")
print(f"Ground truth:  s = {ground_truth_s:.8f} m")
print(f"Computed check: s = {s_check:.8f} m")
print()

# Verification
if abs(s - ground_truth_s) < 1e-4:
    print("PASS: The AI-generated code produces the correct drawdown value.")
else:
    print("FAIL: The result does not match the ground truth. Check the implementation.")
    print(f"  Difference: {abs(s - ground_truth_s):.8f} m")

### Step 4: Verification checklist

Go through this checklist for the AI-generated code above:

- [ ] **Runs without errors**: Did the code execute without exceptions?
- [ ] **Matches ground truth**: Does the output match `s = 1.72420422 m` (within a reasonable tolerance)?
- [ ] **Domain review**: Are the units consistent? Is the drawdown value physically reasonable (positive, less than aquifer thickness)?
- [ ] **Code quality**: Is the code readable? Does it use appropriate variable names? Is the docstring accurate?

### Prompt Log

Record your prompt-writing experience below:

In [ ]:
# Prompt Log
# ----------
# Prompt version 1: (paste your first attempt here)
#
# Result: (did it work? what was wrong?)
#
# Prompt version 2 (if needed): (paste revision here)
#
# Number of iterations to reach correct result: 
#
# What did you learn about writing prompts?

---

## 6. Responsible AI Use in Scientific Computing

As USGS scientists and government employees, we have additional responsibilities when using AI tools. This section covers attribution, reproducibility, data sensitivity, human review, and provides practical guidance for professional use.

### 6.1 Attribution and Citation of AI-Generated Code

When AI tools contribute to your work, proper attribution is both an ethical obligation and a practical necessity for reproducibility.

**Why attribution matters:**
- USGS publications require clear documentation of methods and tools used
- Colleagues who maintain your code need to know which parts were AI-generated (and may need extra scrutiny)
- Attribution helps track which AI tool and version produced the code, which matters when debugging later

**How to attribute AI-generated code:**

1. **Inline comments** — Mark AI-generated sections directly in the code:

```python
# Generated by [AI tool name], [date]
# Prompt: "Write a function to compute baseflow index from daily streamflow"
# Reviewed and modified by [your name] on [date]
def compute_baseflow_index(streamflow):
    ...
```

2. **README or methods section** — In project documentation or publication methods sections, note that AI tools were used and describe the review process.

3. **Version tracking** — Record the AI tool name and version (e.g., "GitHub Copilot, March 2025") because AI models change over time and may produce different output in the future.

> **USGS context:** There is no single bureau-wide policy on AI code citation yet, but the principles of scientific transparency apply. When in doubt, over-document rather than under-document.

### 6.2 Reproducibility Challenges with Non-Deterministic Output

A core principle of scientific computing is **reproducibility** — given the same inputs, you should get the same outputs. AI coding assistants challenge this in several ways:

**The problem:**
- Ask the same AI tool the same question twice, and you may get different code each time
- AI models are updated regularly; a prompt that works today may produce different (or worse) output next month
- Different AI tools (Copilot vs. Claude vs. Amazon Q) will produce different implementations of the same task

**What this means for your work:**
- You cannot cite "AI-generated" as a reproducible method step the way you cite a published algorithm
- A colleague cannot reproduce your analysis by re-running the same prompt

**Mitigation strategies:**

| Strategy | How it helps |
|----------|-------------|
| **Save the final code, not the prompt** | The code you actually ran *is* reproducible |
| **Version control everything** | Commit AI-generated code to Git just like any other code |
| **Write tests** | Tests verify behavior regardless of how the code was written |
| **Document the prompt and tool version** | Provides context even if exact reproduction is not possible |
| **Consider Pinning dependencies** | Ensure the scientific libraries (NumPy, SciPy, pandas) are version-locked unless you plan to stay on top of updates |

The bottom line: **the code you commit and test is your reproducible record**, not the AI interaction that produced it.

### 6.3 Data Sensitivity: Local vs. Cloud-Based AI Tools

When hen you use an AI coding assistant, your prompts and code context may be transmitted to external servers. You need to understand what data you are sharing and whether that is appropriate.

#### How AI tools handle your data

| Tool Type | Where data is processed | Data leaves your machine? |
|-----------|------------------------|---------------------------|
| **Cloud-based** (ChatGPT, Claude web, Copilot) | External servers operated by a third party | **Yes** — your prompts, code snippets, and context are sent to remote servers |
| **Local models** (Ollama, local LLMs) | Your own machine | **No** — everything stays on your computer |
| **IDE extensions** (Copilot in VS Code, Amazon Q) | Varies — check the specific tool's data policy | **Usually yes** — code context is sent for completion |

#### USGS data classification considerations

Before using any AI tool with your work data, consider what information is in your code and prompts:

**Generally appropriate for cloud-based AI tools:**
- Published, publicly available data (data already in NWIS, ScienceBase, etc.)
- Generic code questions ("How do I read a CSV with pandas?")
- Code using only public libraries and public datasets
- Learning exercises like this course

**Use caution — consult your supervisor or IT security:**
- Pre-publication data or preliminary results not yet cleared for release
- Code that contains file paths, server names, or internal network details
- Analysis related to sensitive species locations or critical infrastructure
- Anything involving Controlled Unclassified Information (CUI)

**Do NOT send to cloud-based AI tools:**
- Personally identifiable information (PII) of colleagues or the public
- Credentials, API keys, passwords, or access tokens
- Data under active legal hold or FOIA processing
- Classified or restricted-access information

> **Practical tip:** When in doubt, use a **local** AI tool or strip sensitive details from your prompt before sending it to a cloud service. You can often describe the *structure* of your data ("a DataFrame with columns for site_id, date, and discharge") without including the actual values.

### 6.4 Human Review Requirements for Scientific Conclusions

AI tools can write code, but they cannot evaluate whether the scientific conclusions drawn from that code are valid. **Human expert review is non-negotiable** for any work that informs decisions.

**Why AI cannot replace human review:**
- AI does not understand the physical system being modeled
- AI cannot assess whether results are consistent with field observations
- AI cannot judge whether assumptions in the code match the real-world conditions
- AI cannot determine if a result is policy-relevant or could affect public safety

**What human review should cover:**

1. **Physical plausibility** — Are the computed values within physically reasonable ranges? (e.g., Is the computed hydraulic conductivity within the range for the known geology?)

2. **Consistency with observations** — Do model results align with available field data?

3. **Sensitivity to assumptions** — Would different reasonable assumptions change the conclusion?

4. **Appropriate methods** — Is the analytical or numerical method appropriate for the problem? (AI may apply a method it has seen frequently, even if it is not the best choice for your specific situation.)

5. **Peer review** — For any work that will be published or used in decision-making, another domain expert should review both the code and the results.

> **USGS context:** The Fundamental Science Practices (FSP) require that scientific information products undergo appropriate review. AI-generated code used in those products should be reviewed with the same rigor as any other analytical method.

### 6.5 Case Studies

The following two case studies illustrate how AI tools can get it wrong — or get it right — in hydrology work.

#### Case Study 1: AI Produces Plausible but Wrong Baseflow Separation

**Scenario:** A hydrologist needs to estimate baseflow from a daily streamflow record for a water allocation analysis. They ask an AI assistant:

> *"Write a Python function to separate baseflow from a daily streamflow time series using a recursive digital filter."*

**What the AI produced:**

The AI generated a clean, well-documented function implementing a single-pass Lyne-Hollick recursive digital filter. The code ran without errors and produced a smooth baseflow curve that *looked* reasonable when plotted.

In [ ]:
import numpy as np

# AI-GENERATED CODE (contains a subtle error!)
def baseflow_separation_ai(streamflow, alpha=0.925):
    """Separate baseflow from streamflow using a recursive digital filter.
    
    Parameters
    ----------
    streamflow : array-like
        Daily streamflow values [L^3/T]
    alpha : float
        Filter parameter (default 0.925)
    
    Returns
    -------
    baseflow : numpy.ndarray
        Estimated baseflow [L^3/T]
    """
    Q = np.array(streamflow, dtype=float)
    n = len(Q)
    quickflow = np.zeros(n)
    
    # Single-pass filter (should be THREE passes for Lyne-Hollick)
    quickflow[0] = Q[0] / 2.0
    for i in range(1, n):
        quickflow[i] = alpha * quickflow[i-1] + ((1 + alpha) / 2.0) * (Q[i] - Q[i-1])
        quickflow[i] = max(0, quickflow[i])  # quickflow cannot be negative
    
    baseflow = Q - quickflow
    # Ensure baseflow is not greater than streamflow
    baseflow = np.minimum(baseflow, Q)
    baseflow = np.maximum(baseflow, 0)
    
    return baseflow

**What went wrong:**

The AI implemented only a **single-pass** filter. The standard Lyne-Hollick method uses **three passes** (forward, backward, forward) to avoid phase distortion and produce a more physically realistic baseflow estimate. The single-pass version:

- **Overestimates baseflow** during rising limbs of storm events
- **Underestimates baseflow** during recession periods
- Produces a baseflow index (BFI) that is systematically biased

**Why this matters for water allocation:**

If this baseflow estimate were used in a water allocation analysis, the overestimated baseflow could lead to:
- Overallocation of water rights based on inflated "reliable" baseflow
- Underestimation of the flashy (storm-driven) component of streamflow
- Incorrect assessment of minimum environmental flows

The result *looked* plausible — the baseflow curve was smooth, always positive, and always less than total streamflow. A quick visual check would not catch this error. Only comparison against a validated implementation or published BFI values for the same gage would reveal the bias.

**Lesson:** AI-generated code can produce results that pass basic sanity checks but contain methodological errors that affect scientific conclusions. For analyses that inform management decisions, always validate against established implementations or published reference values.

#### Case Study 2: AI Correctly Accelerates Routine NWIS Data Download

**Scenario:** A hydrologist needs to download daily streamflow data from USGS NWIS for 15 gaging stations and format it into a single pandas DataFrame for analysis. This is a routine data retrieval task with well-defined inputs and outputs.

**The prompt used:**

> *"Write a Python function using the dataretrieval package to download daily mean streamflow (parameter code 00060, stat code 00003) from NWIS for a list of site numbers between a start date and end date. Return a single pandas DataFrame with columns: site_no, datetime, and discharge_cms. Convert discharge from cfs to cms using the factor 0.0283168. Handle cases where a site has no data by skipping it and printing a warning."*

In [ ]:
# AI-GENERATED CODE (this one is correct!)
# Note: This code requires the 'dataretrieval' package and internet access.
# It is shown here for illustration; we do not run it in class.

def download_nwis_streamflow(site_numbers, start_date, end_date):
    """Download daily mean streamflow from USGS NWIS for multiple sites.
    
    Parameters
    ----------
    site_numbers : list of str
        USGS site numbers (e.g., ['05407000', '05408000'])
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format
    
    Returns
    -------
    pandas.DataFrame
        DataFrame with columns: site_no, datetime, discharge_cms
    """
    import pandas as pd
    from dataretrieval import nwis
    
    CFS_TO_CMS = 0.0283168
    all_data = []
    
    for site in site_numbers:
        try:
            df, _ = nwis.get_dv(
                sites=site,
                parameterCd='00060',
                statCd='00003',
                start=start_date,
                end=end_date
            )
            if df.empty:
                print(f"Warning: No data returned for site {site}")
                continue
            
            # Rename and convert
            result = pd.DataFrame({
                'site_no': site,
                'datetime': df.index,
                'discharge_cms': df['00060_Mean'].values * CFS_TO_CMS
            })
            all_data.append(result)
        except Exception as e:
            print(f"Warning: Could not retrieve data for site {site}: {e}")
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    else:
        return pd.DataFrame(columns=['site_no', 'datetime', 'discharge_cms'])

**Why this worked well:**

This task was a good fit for AI assistance because:

- ✅ **Well-defined API**: The `dataretrieval` package has clear, documented functions
- ✅ **Routine pattern**: Downloading and reformatting data is a common pattern the AI has seen many times
- ✅ **Specific prompt**: The prompt included parameter codes, conversion factors, and error handling requirements
- ✅ **Easy to verify**: You can spot-check a few values against NWIS Web to confirm correctness
- ✅ **Low risk**: A formatting error in data download code is generally easy to catch

**Time saved:** Writing this function manually would take 15-20 minutes including looking up the `dataretrieval` API. The AI produced a working version in seconds. The hydrologist spent 5 minutes reviewing the code and spot-checking output — a net time savings of 10-15 minutes.

**Lesson:** AI tools excel at routine data retrieval and formatting tasks where the inputs and outputs are well-defined, the APIs are well-documented, and the results are easy to verify. This is exactly the kind of task where AI assistance provides clear value.

### 6.6 Professional Use Checklist

Use this checklist when deciding whether and how to use an AI coding assistant for a task in your professional work. Consider printing or bookmarking this as a quick reference.

---

#### 📋 AI-Assisted Coding: Decision Checklist for USGS Scientists

**Before using an AI tool, ask:**

☐ **Data sensitivity** — Does my prompt or code context contain sensitive, pre-publication, or restricted data?  
&nbsp;&nbsp;&nbsp;&nbsp;→ If yes: use a local AI tool, or remove sensitive details from the prompt  
&nbsp;&nbsp;&nbsp;&nbsp;→ If unsure: consult your supervisor or IT security contact

☐ **Task suitability** — Is this task a good fit for AI assistance?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Good fit: boilerplate code, data formatting, plot styling, API lookups  
&nbsp;&nbsp;&nbsp;&nbsp;→ Poor fit: novel scientific methods, domain-specific logic, policy-relevant calculations

☐ **Verification plan** — How will I verify the AI-generated code is correct?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Do I have a ground truth value, analytical solution, or reference implementation?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Can I write a test to check the output?

**While using an AI tool:**

☐ **Prompt quality** — Did I provide specific inputs, outputs, units, and constraints?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Use the prompt patterns from Section 4 of this notebook

☐ **Critical reading** — Am I reading the generated code line by line, or just checking if it runs?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Running without errors ≠ correct - the Athens test is not enough!

**After using an AI tool:**

☐ **Verification** — Did I check the output against known values or expected behavior?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Use the verification checklist from Section 5

☐ **Attribution** — Did I document that AI was used, which tool, and what was reviewed/modified?  
&nbsp;&nbsp;&nbsp;&nbsp;→ Add inline comments and note in project documentation

☐ **Reproducibility** — Is the final code committed to version control with tests?  
&nbsp;&nbsp;&nbsp;&nbsp;→ The code (not the prompt) is the reproducible artifact

☐ **Peer review** — If this code supports a publication or decision, has another domain expert reviewed it?  
&nbsp;&nbsp;&nbsp;&nbsp;→ AI-generated code should receive the same review rigor as manually written code

---

> **Remember:** AI is a tool in your workflow, not a replacement for your expertise. The scientific judgment, domain knowledge, and professional responsibility remain yours.

## Summary

In this notebook we covered:

1. **How AI coding assistants work** — they predict likely code based on patterns, not understanding
2. **Strengths** — boilerplate, syntax, pattern completion
3. **Limitations** — no domain knowledge, hallucination, cannot verify physics
4. **Three failure examples** — unit conversion errors, equation sign errors, missing physical constraints
5. **Four prompt patterns** — data transformations, equations, plots, file I/O
6. **Verification practice** — always check AI output against ground truth
7. **Responsible AI use** — attribution, reproducibility, data sensitivity for USGS work, and human review requirements
8. **Case studies** — how AI can go wrong (baseflow separation) and go right (NWIS data download)
9. **Professional use checklist** — a quick reference for deciding when and how to use AI tools

**Key takeaway:** AI coding assistants are powerful tools for accelerating routine programming tasks, but they require your domain expertise to produce scientifically correct results. As USGS scientists, we have additional responsibilities around data sensitivity, reproducibility, and ensuring that AI-assisted work meets the same standards as any other scientific product. Always verify, always document, always review.